In [1]:
import pandas as pd

# Load datasets 
phishing_df = pd.read_csv('PhiUSIIL_Phishing_URL_Dataset.csv')
tranco_df = pd.read_csv('tranco_KW6WW.csv')

In [2]:
# Look at each one
print("=== PHISHING DATASET ===")
print(phishing_df.shape)
print(phishing_df.head())
print(phishing_df.columns.tolist())

print("\n=== TRANCO DATASET ===")
print(tranco_df.shape)
print(tranco_df.head())
print(tranco_df.columns.tolist())

=== PHISHING DATASET ===
(235795, 56)
     FILENAME                                 URL  URLLength  \
0  521848.txt    https://www.southbankmosaics.com         31   
1   31372.txt            https://www.uni-mainz.de         23   
2  597387.txt      https://www.voicefmradio.co.uk         29   
3  554095.txt         https://www.sfnmjournal.com         26   
4  151578.txt  https://www.rewildingargentina.org         33   

                       Domain  DomainLength  IsDomainIP  TLD  \
0    www.southbankmosaics.com            24           0  com   
1            www.uni-mainz.de            16           0   de   
2      www.voicefmradio.co.uk            22           0   uk   
3         www.sfnmjournal.com            19           0  com   
4  www.rewildingargentina.org            26           0  org   

   URLSimilarityIndex  CharContinuationRate  TLDLegitimateProb  ...  Pay  \
0               100.0              1.000000           0.522907  ...    0   
1               100.0              0.666

In [3]:
# How many phishing vs legitimate?
print(phishing_df['label'].value_counts())

# Look at just the URL column
print("\n=== PHISHING URLs ===")
print(phishing_df[phishing_df['label']==1]['URL'].head(10).to_string())

print("\n=== LEGITIMATE URLs ===")
print(phishing_df[phishing_df['label']==0]['URL'].head(10).to_string())

label
1    134850
0    100945
Name: count, dtype: int64

=== PHISHING URLs ===
0      https://www.southbankmosaics.com
1              https://www.uni-mainz.de
2        https://www.voicefmradio.co.uk
3           https://www.sfnmjournal.com
4    https://www.rewildingargentina.org
5       https://www.globalreporting.org
6            https://www.saffronart.com
7            https://www.nerdscandy.com
8        https://www.hyderabadonline.in
9                   https://www.aap.org

=== LEGITIMATE URLs ===
11                              http://www.teramill.com
20                          http://www.f0519141.xsph.ru
21                             http://www.shprakserf.gq
27               https://service-mitld.firebaseapp.com/
28                    http://www.kuradox92.lima-city.de
29                          https://liuy-9a930.web.app/
31    https://ipfs.io/ipfs/qmrvvyr84esa2assw9vvwupqj...
32             http://att-103731-107123.weeblysite.com/
34                  https://hidok4f8zl.firebasea

In [4]:
# Flip the labels so 1 = phishing, 0 = legitimate
phishing_df['label'] = phishing_df['label'].map({1: 0, 0: 1})

# Verify the fix
print(phishing_df['label'].value_counts())

print("\n=== PHISHING URLs ===")
print(phishing_df[phishing_df['label']==1]['URL'].head(10).to_string())

print("\n=== LEGITIMATE URLs ===")
print(phishing_df[phishing_df['label']==0]['URL'].head(10).to_string())

label
0    134850
1    100945
Name: count, dtype: int64

=== PHISHING URLs ===
11                              http://www.teramill.com
20                          http://www.f0519141.xsph.ru
21                             http://www.shprakserf.gq
27               https://service-mitld.firebaseapp.com/
28                    http://www.kuradox92.lima-city.de
29                          https://liuy-9a930.web.app/
31    https://ipfs.io/ipfs/qmrvvyr84esa2assw9vvwupqj...
32             http://att-103731-107123.weeblysite.com/
34                  https://hidok4f8zl.firebaseapp.com/
37                                 http://www.ooguy.com

=== LEGITIMATE URLs ===
0      https://www.southbankmosaics.com
1              https://www.uni-mainz.de
2        https://www.voicefmradio.co.uk
3           https://www.sfnmjournal.com
4    https://www.rewildingargentina.org
5       https://www.globalreporting.org
6            https://www.saffronart.com
7            https://www.nerdscandy.com
8        https:/

In [5]:
def is_https(url):
    return 1 if url.startswith('https') else 0

# Test it
print(is_https('https://www.netflix.com'))   # should print 1
print(is_https('http://att-103731.weeblysite.com'))  # should print 0

1
0


In [6]:
def url_length(url):
    return len(url)

print(url_length('https://www.netflix.com'))
print(url_length('http://att-103731.weeblysite.com'))  

23
32


In [7]:
def digit_ratio_in_domain(url):
    from urllib.parse import urlparse
    domain = urlparse(url).netloc  # extracts just the domain part
    domlen = len(domain)
    
    count = 0
    for i in domain:        # loop over the string, not the number
        if i.isdigit():     # check each character
            count += 1      # add 1 every time you find a digit
    
    return count / domlen   # return ratio AFTER the loop

    print(f"domain: {domain}, digits: {count}, length: {domlen}")  # debug line
    return count / domlen
 
print(digit_ratio_in_domain ('http://www.f0519141.xsph.ru'))
print(digit_ratio_in_domain('http://www.netflix.com'))      

0.35
0.0


In [8]:
phishing_df['is_https'] = phishing_df['URL'].apply(is_https)
phishing_df['url_length'] = phishing_df['URL'].apply(url_length)
phishing_df['digit_ratio'] = phishing_df['URL'].apply(digit_ratio_in_domain)

# Now look at the averages by label
print(phishing_df.groupby('label')[['is_https', 'url_length', 'digit_ratio']].mean())

       is_https  url_length  digit_ratio
label                                   
0      1.000000   27.228610     0.003042
1      0.487354   46.238774     0.068511


In [12]:
SUSPICIOUS_TLDS = ['.ru', '.gq', '.ml', '.tk', '.ga', '.cf', '.xyz']

def suspicious_tld(url):
    for tld in SUSPICIOUS_TLDS:
        if url.endswith(tld):  
            return 1
    return 0

print(suspicious_tld('http://www.f0519141.xsph.ru'))
print(suspicious_tld('https://www.netflix.com'))  

1
0


In [13]:
phishing_df['suspicious_tld'] = phishing_df['URL'].apply(suspicious_tld)

print(phishing_df.groupby('label')[['is_https', 'url_length', 'digit_ratio', 'suspicious_tld']].mean())

       is_https  url_length  digit_ratio  suspicious_tld
label                                                   
0      1.000000   27.228610     0.003042        0.006867
1      0.487354   46.238774     0.068511        0.058061


In [14]:
print(phishing_df.shape)
print(phishing_df[['is_https', 'url_length', 'digit_ratio', 'suspicious_tld', 'label']].head())

(235795, 60)
   is_https  url_length  digit_ratio  suspicious_tld  label
0         1          32          0.0               0      0
1         1          24          0.0               0      0
2         1          30          0.0               0      0
3         1          27          0.0               0      0
4         1          34          0.0               0      0


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [17]:
# Define our features and target
X = phishing_df[['is_https', 'url_length', 'digit_ratio', 'suspicious_tld']]
y = phishing_df['label']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

Training set: (188636, 4)
Testing set: (47159, 4)


In [19]:
# Build the model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
# n_estimators - 100 trees in the forest
print("Model trained!")

Model trained!


In [20]:
# Make predictions on the test set
y_pred = rf.predict(X_test)

# Print the results
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))

              precision    recall  f1-score   support

  Legitimate       0.93      0.99      0.96     27035
    Phishing       0.98      0.90      0.94     20124

    accuracy                           0.95     47159
   macro avg       0.95      0.94      0.95     47159
weighted avg       0.95      0.95      0.95     47159



In [22]:
import pandas as pd
import matplotlib.pyplot as plt

# Get feature importances
features = ['is_https', 'url_length', 'digit_ratio', 'suspicious_tld']
importances = rf.feature_importances_

# Display them
importance_df = pd.DataFrame({'feature': features, 'importance': importances})
importance_df = importance_df.sort_values('importance', ascending=False)
print(importance_df)

          feature  importance
0        is_https    0.472507
1      url_length    0.345196
2     digit_ratio    0.176046
3  suspicious_tld    0.006251


In [23]:
import pandas as pd

# Can you craft a URL that IS phishing but fools the model?
test_urls = pd.DataFrame({
    'is_https': [1],           # use https to look legitimate
    'url_length': [25],        # keep it short
    'digit_ratio': [0.0],      # no digits
    'suspicious_tld': [0]      # clean TLD
})

print(rf.predict(test_urls))
print(rf.predict_proba(test_urls))

[0]
[[0.96187198 0.03812802]]


In [25]:
#model is blind to typosquatting. [0] — the model predicted legitimate. You fooled it.
# [[0.96 0.04]] — it's 96% confident it's legitimate. Not just wrong, but confidently wrong.
# What the code did — you manually crafted a fake URL profile:
# HTTPS 
# Short URL 
# No digits 
# Clean TLD 
#And the model said "looks clean to me!" But a real attacker could easily make a phishing URL that ticks all those boxes. Like:
# https://www.netf1ix.com
# That URL is:
# HTTPS 
# Short 
# No digits in domain 
# .com TLD 

In [31]:
import math
from urllib.parse import urlparse
from collections import Counter

def path_entropy(url):
    # Step 1: extract just the path from the url
    path = urlparse(url).path
    
    # Step 2: if path is empty or just '/' return 0
    if path == "" or path == "/":
        return 0
    
    # Step 3: count frequency of each character in the path
    counts = Counter(path)
    total = len(path)
    
    entropy = 0
    for count in counts.values():
        probability = count / total
        entropy += -(probability * math.log2(probability))
    
    # Step 6: return it
    return entropy

# Test it
print(path_entropy('https://google.com/search'))                    # should be low
print(path_entropy('https://x9k2.com/aK93mZ/ryuie/playlist-10928')) # should be high

2.807354922057604
4.280394654123195


In [32]:
phishing_df['path_entropy'] = phishing_df['URL'].apply(path_entropy)

print(phishing_df.groupby('label')[['is_https', 'url_length', 'digit_ratio', 'suspicious_tld', 'path_entropy']].mean())

       is_https  url_length  digit_ratio  suspicious_tld  path_entropy
label                                                                 
0      1.000000   27.228610     0.003042        0.006867      0.000000
1      0.487354   46.238774     0.068511        0.058061      0.961434


In [33]:
# Retrain with new feature added
X = phishing_df[['is_https', 'url_length', 'digit_ratio', 'suspicious_tld', 'path_entropy']]
y = phishing_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))

              precision    recall  f1-score   support

  Legitimate       0.94      0.99      0.97     27035
    Phishing       0.99      0.92      0.95     20124

    accuracy                           0.96     47159
   macro avg       0.97      0.96      0.96     47159
weighted avg       0.96      0.96      0.96     47159



In [34]:
features = ['is_https', 'url_length', 'digit_ratio', 'suspicious_tld', 'path_entropy']
importances = rf.feature_importances_

importance_df = pd.DataFrame({'feature': features, 'importance': importances})
importance_df = importance_df.sort_values('importance', ascending=False)
print(importance_df)

          feature  importance
0        is_https    0.453579
1      url_length    0.234554
2     digit_ratio    0.182194
4    path_entropy    0.121466
3  suspicious_tld    0.008207
